In [ ]:
#| default_exp core

## FastCF

In [ ]:
import hashlib, json
from fastcore.utils import *
from fastcore.net import urlread
from fastspec.oapi import SpecParser, OpenAPIClient
from fastcore.apisurface import snake
from collections import defaultdict

In [ ]:
import httpx
from collections import Counter

In [ ]:
tok = os.environ['CLOUDFLARE_API_TOKEN']
acctid = os.getenv('CLOUDFLARE_ACCOUNT_ID')
email = os.getenv('CLOUDFLARE_EMAIL')
api_key = os.getenv('CLOUDFLARE_API_KEY')

In [ ]:
base = 'https://api.cloudflare.com/client/v4'

In [ ]:
uhdrs = {'Authorization': f'Bearer {tok}'}
r = httpx.get(f'{base}/user/tokens/verify', headers=uhdrs)
r.json()['success']

True

In [ ]:
if email and api_key:
    ghdrs = {'X-Auth-Email': email, 'X-Auth-Key': api_key, 'Content-Type': 'application/json'}
    r = httpx.get(f'{base}/user', headers=ghdrs)
    print(r.json()['success'])

True


In [ ]:
r = httpx.get(f'{base}/zones', headers=uhdrs)
[(z['name'], z['id'][:8]) for z in r.json()['result'][:1]]

[('aaa.as', '6afde8dc')]

## CloudflareApi

Cloudflare's operationIds are machine artifacts (`zones-get` vs `zones-0-get` — the `-0-` is a generator's dedup counter), so rather than cleaning them up we name every op mechanically from its verb and path: the group is the path's non-parameter segments, nested (`zones.dns_records`), and the method is the HTTP verb — except that when a group has both a collection GET and a single-item GET they become `list` and `get`. Two fixups keep the whole surface collision-free: sibling paths differing only by an interior parameter gain a `_by_<param>` suffix, and an action segment that would shadow an op folds into its parent as `<segment>_<verb>`. So `GET /zones` is `zones.list`, `GET /zones/{zone_id}` is `zones.get`, and `GET /user/tokens/verify` is `user.tokens.verify.get` — a user who knows the URL knows the call.

In [ ]:
def _canon(verb, path): return verb.lower(), tuple(snake(s.strip('{}')) if s.startswith('{') else snake(s) for s in path.strip('/').split('/'))

def _op_parts(path):
    "Snaked non-param segments, collection flag (path ends on a non-param segment), and param names"
    segs = path.strip('/').split('/')
    return [snake(s) for s in segs if not s.startswith('{')], not segs[-1].startswith('{'), [s.strip('{}') for s in segs if s.startswith('{')]

In [ ]:
def cf_names(spec):
    "Map each op's canonical (verb, snaked-segments) key to (group_segments, op_name), collision-free across the surface"
    ops = {}
    for p, item in spec['paths'].items():
        for v in item:
            if v in ('get','post','put','patch','delete','head','options'): ops.setdefault(_canon(v,p), p)
    gets = defaultdict(set)
    for (v,_),p in ops.items():
        g, coll, _ = _op_parts(p)
        if v=='get': gets[tuple(g)].add(coll)
    res = {}
    for k,p in ops.items():
        v = k[0]
        g, coll, ps = _op_parts(p)
        nm = 'list' if v=='get' and coll and len(gets[tuple(g)])==2 else v
        res[k] = [tuple(g), nm, ps]
    sibs = defaultdict(list)
    for k,(g,n,ps) in res.items(): sibs[(g,n)].append(k)
    for ks in sibs.values():
        if len(ks)==1: continue
        common = set.intersection(*(set(res[k][2]) for k in ks))
        for k in ks:
            extra = [q for q in res[k][2] if q not in common]
            if extra: res[k][1] += '_by_'+'_'.join(extra)
    opnms, kids = defaultdict(set), defaultdict(set)
    for g,n,_ in res.values():
        opnms[g].add(n)
        for i in range(1,len(g)): kids[g[:i]].add(g[i])
    for g,seg in [(g,s) for g in opnms for s in kids[g]&opnms[g]]:
        for k,(og,n,ps) in res.items():
            if og==g+(seg,): res[k] = [g, f'{seg}_{n}', ps]
    return {k:(list(g),n) for k,(g,n,ps) in res.items()}

In [ ]:
def cf_group_func(names):
    "A fastspec `group_func` looking up precomputed `names` by canonical verb+path"
    def f(op_id, path, verb='', path_tags=None, op_tags=None): return names[_canon(verb, path)]
    return f

The parsed surface ships with the package as `cf_spec.py`, fastspec's compact form (`SpecParser.save`): 2.6MB instead of the 8MB raw schema, loaded in ~30ms instead of a ~0.3s parse, with the naming machinery above running only at regeneration time. `build_spec` refreshes it from Cloudflare's published schema; op names can shift when the spec changes, so review that diff before shipping it.

In [ ]:
CF_OPENAPI_URL = 'https://raw.githubusercontent.com/cloudflare/api-schemas/main/openapi.json'

def build_spec(nm='fastcflare/cf_spec.py', url=CF_OPENAPI_URL):
    "Regenerate the compact spec module `cf_spec.py` from Cloudflare's OpenAPI schema."
    schema = dict2obj(json.loads(urlread(url)))
    SpecParser.from_openapi(schema, cf_group_func(cf_names(schema))).save(nm)

def _cf_spec():
    from fastcflare.cf_spec import spec
    return SpecParser.from_dict(spec)

In [ ]:
cli = OpenAPIClient(_cf_spec(), headers={'Authorization': f'Bearer {tok}', 'Content-Type': 'application/json'})
sorted(cli.groups)

['accounts',
 'certificates',
 'internal',
 'ips',
 'live',
 'memberships',
 'organizations',
 'radar',
 'ready',
 'signed_url',
 'system',
 'tenants',
 'user',
 'users',
 'zones']

The naming is only trustworthy if it can never silently collide — `OpGroup` exposes ops with `setattr`, so a duplicate name would shadow its sibling (exactly the bug this design replaces). This walk asserts the built tree is clean: within every group, op names are unique (duplicates allowed only for the same endpoint spelled two ways in the spec) and no op shares a name with a subgroup.

In [ ]:
def _surface_ok(g):
    nms = [o.name for o in g.ops]
    for n,c in Counter(nms).items():
        if c>1: assert len({tuple(snake(s) for s in o.op_spec.path.strip('/').split('/')) for o in g.ops if o.name==n})==1, f'dup op {g.name}.{n}'
    assert not set(nms)&set(g.subgroups), f'op/subgroup shadow in {g.name}: {set(nms)&set(g.subgroups)}'
    for sg in g.subgroups.values(): _surface_ok(sg)

for grp in cli.groups.values(): _surface_ok(grp)
len(cli.groups), len(cli.groups['zones'].subgroups)

(15, 60)

In [ ]:
def _cf_auth(
    token=None, # API token; defaults to `CLOUDFLARE_API_TOKEN` when `api_key` and `email` aren't given
    account_id=None, # Cloudflare account ID; defaults to `CLOUDFLARE_ACCOUNT_ID`
    email=None, # Account email for a global API key; defaults to `CLOUDFLARE_EMAIL`
    api_key=None, # Global API key; defaults to `CLOUDFLARE_API_KEY` when no token is given or set
):
    "Resolve Cloudflare auth headers and `account_id` from arguments and `CLOUDFLARE_*` env vars."
    account_id = account_id or os.getenv('CLOUDFLARE_ACCOUNT_ID')
    if not token:
        if api_key or email:
            api_key = api_key or os.getenv('CLOUDFLARE_API_KEY')
            email = email or os.getenv('CLOUDFLARE_EMAIL')
        elif t := os.getenv('CLOUDFLARE_API_TOKEN'): token = t
        elif k := os.getenv('CLOUDFLARE_API_KEY'): api_key, email = k, os.getenv('CLOUDFLARE_EMAIL')
    hdrs = ({'X-Auth-Email': email, 'X-Auth-Key': api_key} if api_key else {'Authorization': f'Bearer {token}'}) | {'Content-Type': 'application/json'}
    return hdrs, account_id, token, email, api_key

class CloudflareApi:
    "Cloudflare API client supporting user tokens, account tokens, and global API keys"
    def __init__(self,
        token=None, # API token; defaults to `CLOUDFLARE_API_TOKEN` when `api_key` and `email` aren't given
        account_id=None, # Account ID; defaults to `CLOUDFLARE_ACCOUNT_ID`
        email=None, # Account email for a global API key; defaults to `CLOUDFLARE_EMAIL`
        api_key=None, # Global API key; defaults to `CLOUDFLARE_API_KEY` when no token is given or set
        account_token:bool=False, # Is `token` an account-owned token (uses `/accounts/{account_id}/tokens/verify`)?
    ):
        hdrs, account_id, token, email, api_key = _cf_auth(token, account_id, email, api_key)
        store_attr()
        self.cli = OpenAPIClient(_cf_spec(), headers=hdrs, timeout=60.0)
        self.groups = self.cli.groups

    async def verify(self):
        "Check credentials via `/user` (global API key), account token verify (`account_token=True`), or user token verify."
        if self.api_key: return await self.user.get()
        if self.account_token: return await self.accounts.tokens.verify.get(account_id=self.account_id)
        return await self.user.tokens.verify.get()

    def __dir__(self): return super().__dir__() + list(self.groups)
    def _repr_markdown_(self): return "\n".join(f"- {o}" for o in sorted(self.groups))
    def __getattr__(self, k): return self.groups[k] if 'groups' in vars(self) and k in self.groups else stop(AttributeError(k))

In [ ]:
ucf = CloudflareApi()
(await ucf.verify()).result.status

'active'

Listing and fetching zones are now distinct calls: `zones.list` filters the collection, `zones.get` takes a `zone_id`.

In [ ]:
ucf.zones.list

List Zones

Parameters:
- name (str, optional)
- status (str, optional)
- account_id (str, optional)
- account_name (str, optional)
- order (str, optional)
- direction (str, optional)
- page (float, default: 1)
- per_page (float, default: 20)
- match (str, default: 'all')

In [ ]:
(await ucf.zones.list(name='answer.ai')).result[0].name

'answer.ai'

In [ ]:
zid = (await ucf.zones.list()).result[0].id
(await ucf.zones.dns_records.list(zone_id=zid)).result[0].name

'*.aaa.as'

In [ ]:
@patch
async def create_token(
    self:CloudflareApi,
    doms, # Domain names the token may act on
    perm_names, # Permission-group names, such as 'Zone Read' or 'DNS Write'. Names that don't match are skipped
    name, # Name for the new token
    grp='account.zone.', # Resource prefix after `com.cloudflare.api.`
    account_token:bool=False, # Create an account-owned token (nests zone resources under `account_id`)?
    account_id:str=None, # Account ID for account-owned tokens; defaults to `self.account_id`
):
    "Create a user or account-owned API token limited to `doms`. The token's value is shown only in this response."
    pref = 'com.cloudflare.api.'+grp
    acct = account_id or self.account_id
    tok_api = self.accounts.tokens if account_token else self.user.tokens
    pgs = (await (tok_api.permission_groups.get(acct) if account_token else tok_api.permission_groups.get())).result
    perms = [dict(id=p.id) for p in pgs if p.name in perm_names]
    zids = {f'{pref}{(await self.zones.list(name=d)).result[0].id}':'*' for d in doms}
    resources = {f'com.cloudflare.api.account.{acct}': zids} if account_token else zids
    pol = [dict(effect='allow', resources=resources, permission_groups=perms)]
    return await (tok_api.post(acct, name=name, policies=pol) if account_token else tok_api.post(name=name, policies=pol))

In [ ]:
pgs = (await ucf.user.tokens.permission_groups.get()).result
[p.name for p in pgs if 'DNS' in p.name or 'Zone' in p.name][:20]

['Account DNS Settings Read',
 'Account DNS Settings Write',
 'Cloudflare Zero Trust Secure DNS Locations Write',
 'DNS Firewall Read',
 'DNS Firewall Write',
 'DNS Read',
 'DNS View Read',
 'DNS View Write',
 'DNS Write',
 'Zone Custom Asset Read',
 'Zone Custom Asset Write',
 'Zone DNS Settings Read',
 'Zone DNS Settings Write',
 'Zone Read',
 'Zone Security Center Insights Read',
 'Zone Security Center Insights Write',
 'Zone Settings Read',
 'Zone Settings Write',
 'Zone Transform Rules Read',
 'Zone Transform Rules Write']

With two domains, the policy must carry two distinct zone resources — under the old colliding surface the `name` filter was silently dropped, every domain resolved to the same zone, and the dict collapsed them to one:

In [ ]:
tok = await ucf.create_token(['answer.ai', 'aaa.as'], ['Zone Read'], 'fastcflare-test-token')
assert len(tok.result.policies[0].resources)==2
tok.success, tok.result.name

(True, 'fastcflare-test-token')

Clean up the token we just created:

In [ ]:
res = await ucf.user.tokens.delete(tok.result.id)
assert res.success

## R2 tokens and S3 credentials

Cloudflare R2 S3 credentials derive directly from an R2 API token: `access_key_id` is the token ID and `secret_access_key` is the SHA-256 hex digest of the token value. `create_r2_token` creates either an account-owned service token (default) or a user token, scoped to specific buckets or the whole account, and returns both the API response and derived S3 credentials.

In [ ]:
def r2_s3_creds(
    token_id:str, # Cloudflare API token ID (becomes `access_key_id`)
    token_value:str, # Cloudflare API token secret value (SHA-256 hex becomes `secret_access_key`)
    account_id:str=None, # Account ID; defaults to `CLOUDFLARE_ACCOUNT_ID`
    jurisdiction:str='default', # R2 jurisdiction (`'default'`, `'eu'`, `'fedramp'`)
) -> AttrDict:
    "Derive S3-compatible R2 credentials (`access_key_id`, `secret_access_key`, `endpoint`, `region`) from a Cloudflare R2 API token."
    acct = account_id or os.environ['CLOUDFLARE_ACCOUNT_ID']
    host = f'{acct}.r2.cloudflarestorage.com' if jurisdiction=='default' else f'{acct}.{jurisdiction}.r2.cloudflarestorage.com'
    return AttrDict(access_key_id=token_id, secret_access_key=hashlib.sha256(token_value.encode()).hexdigest(),
                    endpoint=f'https://{host}', region='auto')


`r2_s3_creds` derives S3 credentials offline from any existing token pair:

In [ ]:
creds = r2_s3_creds('tok-id', 'tok-val', account_id='acct123')
assert len(creds.secret_access_key) == 64 and creds.endpoint == 'https://acct123.r2.cloudflarestorage.com'
creds.endpoint, creds.region

In [ ]:
@patch
async def create_r2_token(
    self:CloudflareApi,
    name:str, # Name for the new R2 token
    buckets=None, # Bucket name or iterable of bucket names; None grants account-wide R2 access
    write:bool=True, # Grant read/write access (False for read-only)
    account_id:str=None, # Account ID; defaults to `self.account_id`
    account_token:bool=True, # Create an account-owned service token (False for a user-owned token)
    jurisdiction:str='default', # R2 jurisdiction for bucket-scoped resources (`'default'`, `'eu'`, `'fedramp'`)
    expires_on:str=None, # Optional RFC3339 expiration timestamp
) -> AttrDict:
    "Create a Cloudflare R2 API token and return both the API response (`token`) and derived S3 credentials (`s3`)."
    acct = account_id or self.account_id
    if not acct: raise ValueError('account_id is required (pass it or set CLOUDFLARE_ACCOUNT_ID)')
    if isinstance(buckets, str): buckets = [buckets]
    if buckets:
        perm = 'Workers R2 Storage Bucket Item Write' if write else 'Workers R2 Storage Bucket Item Read'
        resources = {f'com.cloudflare.edge.r2.bucket.{acct}_{jurisdiction}_{b}': '*' for b in buckets}
    else:
        perm = 'Workers R2 Storage Write' if write else 'Workers R2 Storage Read'
        resources = {f'com.cloudflare.api.account.{acct}': '*'}
    tok_api = self.accounts.tokens if account_token else self.user.tokens
    pgs = (await (tok_api.permission_groups.get(acct) if account_token else tok_api.permission_groups.get())).result
    perms = [dict(id=p.id) for p in pgs if p.name == perm]
    if not perms: raise ValueError(f'Permission group not found: {perm}')
    kw = dict(name=name, policies=[dict(effect='allow', resources=resources, permission_groups=perms)])
    if expires_on: kw['expires_on'] = expires_on
    res = await (tok_api.post(acct, **kw) if account_token else tok_api.post(**kw))
    return AttrDict(token=res, s3=r2_s3_creds(res.result.id, res.result.value, acct, jurisdiction))

Global API key authentication fills `email` and `account_id` from `CLOUDFLARE_EMAIL` and `CLOUDFLARE_ACCOUNT_ID`:

In [ ]:
if api_key and email and acctid:
    gcf = CloudflareApi(api_key=api_key)
    assert (await gcf.verify()).success

In [ ]:
if api_key and email and acctid:
    r2 = await gcf.create_r2_token('fastcflare-r2-test-token', buckets='fastcflare-test-bucket', write=False)
    try:
        assert r2.token.success and len(r2.s3.secret_access_key) == 64
        r2_cf = CloudflareApi(token=r2.token.result.value, account_token=True)
        assert (await r2_cf.verify()).result.status == 'active'
    finally: assert (await gcf.accounts.tokens.delete(acctid, r2.token.result.id)).success
    r2.s3.endpoint, r2.s3.region


## Export -

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()